# Swedish Riksdag Turnout Backbone

Goal:
Transform SCB municipality-level turnout data for 2010, 2014, 2018 and 2022
into a long-format dataset for modelling the 2014, 2018 and 2022 elections.

In [1]:
# Data handling
import pandas as pd
import numpy as np

print("pandas version:", pd.__version__)

pandas version: 3.0.5


## 1. Prepare Historical Voter Turnout

The turnout dataset contains municipality-level Riksdag election turnout for
2010, 2014, 2018 and 2022.

The historical modelling observations will use:

- 2010 turnout as previous turnout for the 2014 election,
- 2014 turnout as previous turnout for the 2018 election,
- 2018 turnout as previous turnout for the 2022 election.

In [2]:
# Load the raw SCB municipality-level voter-turnout dataset.
#
# encoding="latin1" is required by this particular downloaded CSV.
# skiprows=2 removes the metadata rows above the actual table header.
# Load raw municipal voter-turnout data from the project's raw-data folder.

df = pd.read_csv(
    "../data/raw/ME0104B8_20260906-003101.csv",
    encoding="latin1",
    skiprows=2
)


In [3]:
# Inspect the raw turnout data before transformation.

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (290, 5)

Columns:
['region', '2010', '2014', '2018', '2022']


,region,2010,2014,2018,2022
0,0114 Upplands Väsby,83.2,84.1,84.7,79.5
1,0115 Vallentuna,87.6,89.3,90.0,87.4
2,0117 Österåker,87.3,88.2,89.3,87.3
3,0120 Värmdö,87.7,89.3,90.3,88.4
4,0123 Järfälla,85.0,85.5,86.3,79.4


In [4]:
# The SCB "region" column contains both the municipality code
# and municipality name, for example:
#
# "0114 Upplands Väsby"
#
# Split it only at the first space so municipality names containing
# spaces remain intact.

df[["municipality_code", "municipality_name"]] = (
    df["region"]
    .str.split(" ", n=1, expand=True)
)

In [5]:
# Quality-check the municipality identifiers.

print(
    "Municipality-code lengths:"
)
print(
    df["municipality_code"]
    .str.len()
    .value_counts()
)

df[
    ["municipality_code", "municipality_name"]
].head()

Municipality-code lengths:
municipality_code
4    290
Name: count, dtype: int64


,municipality_code,municipality_name
0,0114,Upplands Väsby
1,0115,Vallentuna
2,0117,Österåker
3,0120,Värmdö
4,0123,Järfälla


In [6]:
# The original combined "region" column is no longer needed
# because municipality code and name now exist separately.

df = df.drop(columns="region")

In [7]:
# Convert election-year columns from wide format to long format.
#
# Before:
# municipality | 2010 | 2014 | 2018 | 2022
#
# After:
# municipality | election_year | turnout_pct

df_long = df.melt(
    id_vars=[
        "municipality_code",
        "municipality_name"
    ],
    value_vars=[
        "2010",
        "2014",
        "2018",
        "2022"
    ],
    var_name="election_year",
    value_name="turnout_pct"
)

In [8]:
# Inspect the reshaped turnout dataset.

print("Long-format shape:", df_long.shape)

df_long.head(10)

Long-format shape: (1160, 4)


,municipality_code,municipality_name,election_year,turnout_pct
0,0114,Upplands Väsby,2010,83.2
1,0115,Vallentuna,2010,87.6
2,0117,Österåker,2010,87.3
3,0120,Värmdö,2010,87.7
4,0123,Järfälla,2010,85.0
5,0125,Ekerö,2010,90.1
6,0126,Huddinge,2010,83.7
7,0127,Botkyrka,2010,78.8
8,0128,Salem,2010,87.3
9,0136,Haninge,2010,83.0


## 2. Create Historical Modelling Backbone

The turnout data are converted to numeric types, sorted chronologically within
each municipality, and used to create previous-election turnout.

The modelling elections are 2014, 2018 and 2022.

Predictor years are aligned two years before each election:

- 2012 → 2014
- 2016 → 2018
- 2020 → 2022

In [9]:
# Convert year and turnout to numeric types.

df_long["election_year"] = df_long["election_year"].astype(int)

df_long["turnout_pct"] = pd.to_numeric(
    df_long["turnout_pct"],
    errors="coerce"
)

print(df_long.dtypes)

municipality_code        str
municipality_name        str
election_year          int64
turnout_pct          float64
dtype: object


In [10]:
# Sort elections chronologically within each municipality.
# This is essential before creating previous-election turnout.

df_long = df_long.sort_values(
    ["municipality_code", "election_year"]
).reset_index(drop=True)

In [11]:
# Create lagged turnout within each municipality.
#
# Example:
# 2014 row receives 2010 turnout
# 2018 row receives 2014 turnout
# 2022 row receives 2018 turnout

df_long["previous_turnout_pct"] = (
    df_long
    .groupby("municipality_code")["turnout_pct"]
    .shift(1)
)

In [12]:
df_long[df_long["municipality_name"] == "Malmö"]

,municipality_code,municipality_name,election_year,turnout_pct,previous_turnout_pct
464,1280,Malmö,2010,79.6,NaN
465,1280,Malmö,2014,79.7,79.6
466,1280,Malmö,2018,82.0,79.7
467,1280,Malmö,2022,77.2,82.0


In [13]:
# Keep only elections used as modelling targets.

df_model = df_long[
    df_long["election_year"].isin([2014, 2018, 2022])
].copy()

In [14]:
# Map each election to the year of its socioeconomic predictors.

predictor_year_map = {
    2014: 2012,
    2018: 2016,
    2022: 2020
}

df_model["predictor_year"] = (
    df_model["election_year"].map(predictor_year_map)
)

In [15]:
# Keep only the variables needed for the historical turnout backbone.

df_model = df_model[
    [
        "municipality_code",
        "municipality_name",
        "election_year",
        "predictor_year",
        "turnout_pct",
        "previous_turnout_pct"
    ]
]

In [16]:
# Quality-check the historical turnout backbone.

print("Shape:", df_model.shape)

print(
    "Unique municipalities:",
    df_model["municipality_code"].nunique()
)

print("\nRows per election:")
print(
    df_model["election_year"]
    .value_counts()
    .sort_index()
)

print("\nMissing values:")
print(
    df_model.isna().sum()
)

Shape: (870, 6)
Unique municipalities: 290

Rows per election:
election_year
2014    290
2018    290
2022    290
Name: count, dtype: int64

Missing values:
municipality_code       0
municipality_name       0
election_year           0
predictor_year          0
turnout_pct             0
previous_turnout_pct    0
dtype: int64


## 3. Prepare Median Household Income

Median disposable household income is reshaped from wide to long format and
matched to each election using the corresponding predictor year.

The predictor years are 2012, 2016, 2020 and 2024.

In [17]:
# Load raw municipal income data from the project's raw-data folder.

income = pd.read_csv(
    "../data/raw/income_data.csv",
    encoding="latin1",
    skiprows=2
)

In [18]:
print(income.columns.tolist())

['region', 'type of household', 'age', '2012', '2016', '2020', '2024']


In [19]:
# Inspect the raw income dataset.

print("Shape:", income.shape)
print("\nColumns:")
print(income.columns.tolist())

income.head()

Shape: (290, 7)

Columns:
['region', 'type of household', 'age', '2012', '2016', '2020', '2024']


,region,type of household,age,2012,2016,2020,2024
0,0114 Upplands Väsby,all households,18+ years,475.3,528.9,525.5,525.3
1,0115 Vallentuna,all households,18+ years,609.5,676.5,666.1,647.3
2,0117 Österåker,all households,18+ years,592.4,647.9,658.6,668.2
3,0120 Värmdö,all households,18+ years,591.2,639.0,642.6,642.6
4,0123 Järfälla,all households,18+ years,499.8,548.4,545.0,523.9


In [20]:
# Split municipality code and name from the SCB region field.

income[["municipality_code", "municipality_name"]] = (
    income["region"]
    .str.split(" ", n=1, expand=True)
)

In [21]:
# Remove columns not required for this project.

income = income.drop(
    columns=["region", "type of household", "age"]
)

In [22]:
# Reshape income from wide format to one row per municipality-year.
income_long = income.melt(
    id_vars=["municipality_code", "municipality_name"],
    value_vars=["2012", "2016", "2020", "2024"],
    var_name="predictor_year",
    value_name="median_income_ksek"
)

In [23]:
# Convert year and income values to numeric types.

income_long["predictor_year"] = (
    income_long["predictor_year"]
    .astype(int)
)

income_long["median_income_ksek"] = pd.to_numeric(
    income_long["median_income_ksek"],
    errors="coerce"
)

print(income_long.dtypes)

municipality_code         str
municipality_name         str
predictor_year          int64
median_income_ksek    float64
dtype: object


In [24]:
# Add income to the historical modelling dataset.

df_model_income = df_model.merge(
    income_long[
        ["municipality_code", "predictor_year", "median_income_ksek"]
    ],
    on=["municipality_code", "predictor_year"],
    how="left"
)

In [25]:
# Quality-check the income merge.

print("Shape:", df_model_income.shape)

print(
    "Unique municipalities:",
    df_model_income["municipality_code"].nunique()
)

print(
    "Missing income values:",
    df_model_income["median_income_ksek"]
    .isna()
    .sum()
)

# One manual municipality check.
df_model_income[
    df_model_income["municipality_name"] == "Malmö"
]

Shape: (870, 7)
Unique municipalities: 290
Missing income values: 0


,municipality_code,municipality_name,election_year,predictor_year,turnout_pct,previous_turnout_pct,median_income_ksek
348,1280,Malmö,2014,2012,79.7,79.6,362.3
349,1280,Malmö,2018,2016,82.0,79.7,412.1
350,1280,Malmö,2022,2020,77.2,82.0,427.3


## 4. Prepare Registered Unemployment

Registered unemployment is prepared for the predictor years 2012, 2016,
2020 and 2024.

Historical values for 2012–2020 and the 2024 data are combined, reshaped
to long format, and matched to the historical modelling dataset using
municipality code and predictor year.

In [26]:
# Load historical unemployment data for 2012, 2016 and 2020 from the raw-data folder.

unemp_old = pd.read_csv(
    "../data/raw/scb_unemployment_2012_2020.csv",
    encoding="latin1",
    skiprows=2
)

In [27]:

# Load unemployment data for 2024 from the raw-data folder.

unemp_new = pd.read_csv(
    "../data/raw/scb_unemployment_2024.csv",
    encoding="latin1",
    skiprows=2
)

In [28]:
# Inspect both raw unemployment datasets.

print("Historical unemployment shape:", unemp_old.shape)
print("2024 unemployment shape:", unemp_new.shape)

print("\nHistorical columns:")
print(unemp_old.columns.tolist())

print("\n2024 columns:")
print(unemp_new.columns.tolist())

display(unemp_old.head())
display(unemp_new.head())

Historical unemployment shape: (290, 7)
2024 unemployment shape: (290, 5)

Historical columns:
['region', 'sex', 'level of education', 'background variable', '2012', '2016', '2020']

2024 columns:
['region', 'sex', 'level of education', 'background variable', '2024']


,region,sex,level of education,background variable,2012,2016,2020
0,0114 Upplands Väsby,men and women,all levels of education,all,12.0,11.9,15.9
1,0115 Vallentuna,men and women,all levels of education,all,6.8,6.1,8.5
2,0117 Österåker,men and women,all levels of education,all,8.4,7.2,9.1
3,0120 Värmdö,men and women,all levels of education,all,8.2,7.3,9.7
4,0123 Järfälla,men and women,all levels of education,all,12.7,13.7,16.2


,region,sex,level of education,background variable,2024
0,0114 Upplands Väsby,men and women,all levels of education,all,13.4
1,0115 Vallentuna,men and women,all levels of education,all,8.0
2,0117 Österåker,men and women,all levels of education,all,8.0
3,0120 Värmdö,men and women,all levels of education,all,7.9
4,0123 Järfälla,men and women,all levels of education,all,14.0


In [29]:
# Split municipality code and name in the historical unemployment data.
unemp_old[["municipality_code", "municipality_name"]]=(
    unemp_old["region"]
     .str.split(" ", n=1, expand=True))

# Do the same for the 2024 unemployment data.
unemp_new[["municipality_code", "municipality_name"]] = (
    unemp_new["region"]
    .str.split(" ", n=1, expand=True)
)


In [30]:
# Keep only the required historical variables.

unemp_old = unemp_old[
    [
        "municipality_code",
        "municipality_name",
        "2012",
        "2016",
        "2020"
    ]
]

# Keep only the required 2024 variables.

unemp_new = unemp_new[
    [
        "municipality_code",
        "municipality_name",
        "2024"
    ]
]

In [31]:
# Confirm that both source datasets contain all 290 municipalities.

print(
    "Historical municipalities:",
    unemp_old["municipality_code"].nunique()
)

print(
    "2024 municipalities:",
    unemp_new["municipality_code"].nunique()
)

Historical municipalities: 290
2024 municipalities: 290


In [32]:
# Add the 2024 unemployment values to the historical unemployment table.
unemployment = unemp_old.merge(
    unemp_new[
        ["municipality_code", "2024"]
    ],
    on="municipality_code",
    how="left"
)

In [33]:
# Quality-check the combined wide-format unemployment table.

print("Shape:", unemployment.shape)

print(
    "Missing 2024 values:",
    unemployment["2024"].isna().sum()
)

unemployment.head()

Shape: (290, 6)
Missing 2024 values: 0


,municipality_code,municipality_name,2012,2016,2020,2024
0,0114,Upplands Väsby,12.0,11.9,15.9,13.4
1,0115,Vallentuna,6.8,6.1,8.5,8.0
2,0117,Österåker,8.4,7.2,9.1,8.0
3,0120,Värmdö,8.2,7.3,9.7,7.9
4,0123,Järfälla,12.7,13.7,16.2,14.0


In [34]:
# Reshape unemployment from wide to long format.

unemployment_long = unemployment.melt(
    id_vars=["municipality_code", "municipality_name"],
    value_vars=["2012", "2016", "2020", "2024"],
    var_name="predictor_year",
    value_name="registered_unemployed_pct"
)

In [35]:
# Convert predictor year and unemployment values to numeric types.

unemployment_long["predictor_year"] = (
    unemployment_long["predictor_year"].astype(int)
)
unemployment_long["registered_unemployed_pct"] = pd.to_numeric(
    unemployment_long["registered_unemployed_pct"],
    errors="coerce"
)

In [36]:
# Quality-check the long-format unemployment dataset.

print("Shape:", unemployment_long.shape)

print("\nRows per predictor year:")
print(
    unemployment_long["predictor_year"]
    .value_counts()
    .sort_index()
)

print(
    "\nMissing unemployment values:",
    unemployment_long["registered_unemployed_pct"]
    .isna()
    .sum()
)

# One manual municipality check.
unemployment_long[
    unemployment_long["municipality_name"] == "Malmö"
]

Shape: (1160, 4)

Rows per predictor year:
predictor_year
2012    290
2016    290
2020    290
2024    290
Name: count, dtype: int64

Missing unemployment values: 0


,municipality_code,municipality_name,predictor_year,registered_unemployed_pct
116,1280,Malmö,2012,18.8
406,1280,Malmö,2016,19.8
696,1280,Malmö,2020,21.2
986,1280,Malmö,2024,18.2


In [37]:
# Add unemployment to the historical modelling dataset.

master_df = df_model_income.merge(
    unemployment_long[
        [
            "municipality_code",
            "predictor_year",
            "registered_unemployed_pct"
        ]
    ],
    on=["municipality_code", "predictor_year"],
    how="left"
)

In [38]:
# Quality-check the historical dataset after adding unemployment.

print("Shape:", master_df.shape)

print(
    "Unique municipalities:",
    master_df["municipality_code"].nunique()
)

print(
    "Missing unemployment values:",
    master_df["registered_unemployed_pct"]
    .isna()
    .sum()
)

print(
    "Duplicate municipality-election rows:",
    master_df.duplicated(
        subset=[
            "municipality_code",
            "election_year"
        ]
    ).sum()
)

# One manual municipality check.
master_df[
    master_df["municipality_name"] == "Malmö"
]

Shape: (870, 8)
Unique municipalities: 290
Missing unemployment values: 0
Duplicate municipality-election rows: 0


,municipality_code,municipality_name,election_year,predictor_year,turnout_pct,previous_turnout_pct,median_income_ksek,registered_unemployed_pct
348,1280,Malmö,2014,2012,79.7,79.6,362.3,18.8
349,1280,Malmö,2018,2016,82.0,79.7,412.1,19.8
350,1280,Malmö,2022,2020,77.2,82.0,427.3,21.2


## 5. Prepare Higher-Education Indicator

The education dataset contains population counts by municipality, age group,
sex, education level and year.

For this project, the indicator is defined as the percentage of the population
aged 25–64 with post-secondary education among people whose education level
is known.

The calculation is:

\[
\text{Higher education \%}
=
\frac{\text{Population with post-secondary education}}
{\text{Population with known education level}}
\times 100
\]

Counts are aggregated across the included age groups and both sexes for each
municipality and predictor year.

## 5. Prepare Higher-Education Indicator

The education dataset contains population counts by municipality, age group,
sex, education level and year.

For this project, the indicator is defined as the percentage of the population
aged 25–64 with post-secondary education among people whose education level
is known.

The calculation is:

\[
\text{Higher education \%}
=
\frac{\text{Population with post-secondary education}}
{\text{Population with known education level}}
\times 100
\]

Counts are aggregated across the included age groups and both sexes for each
municipality and predictor year.

In [39]:
# Load municipal higher-education data from the raw-data folder.

education = pd.read_csv(
    "../data/raw/scb_education_2012_2024.csv",
    encoding="latin1",
    skiprows=2
)

In [40]:
# Inspect the raw education dataset.

print("Shape:", education.shape)

print("\nColumns:")
print(education.columns.tolist())

education.head()

Shape: (18560, 8)

Columns:
['region', 'level of education', 'age', 'sex', '2012', '2016', '2020', '2024']


,region,level of education,age,sex,2012,2016,2020,2024
0,0114 Upplands Väsby,primary and secondary education less than 9 ye...,25-34 years,men,64,83,79,82
1,0114 Upplands Väsby,primary and secondary education less than 9 ye...,25-34 years,women,76,72,58,50
2,0114 Upplands Väsby,primary and secondary education less than 9 ye...,35-44 years,men,67,104,131,126
3,0114 Upplands Väsby,primary and secondary education less than 9 ye...,35-44 years,women,71,79,108,106
4,0114 Upplands Väsby,primary and secondary education less than 9 ye...,45-54 years,men,59,81,98,128


In [41]:
# Split municipality code and municipality name from the SCB region field.
education[["municipality_code", "municipality_name"]] = (
    education["region"]
    .str.split(" ", n=1, expand=True)
)

In [42]:
# Reshape education counts from wide to long format.

education_long = education.melt(
    id_vars=[
        "municipality_code",
        "municipality_name",
        "level of education",
        "age",
        "sex"
    ],
    value_vars=["2012", "2016", "2020", "2024"],
    var_name="predictor_year",
    value_name="population_count"
)

In [43]:
# Convert year and population counts to numeric types.

education_long["predictor_year"] = (
    education_long["predictor_year"]
    .astype(int)
)

education_long["population_count"] = pd.to_numeric(
    education_long["population_count"],
    errors="coerce"
)


# Quality-check the reshaped education data.

print("Shape:", education_long.shape)

print("\nMissing values:")
print(education_long.isna().sum())

education_long.head()

Shape: (74240, 7)

Missing values:
municipality_code     0
municipality_name     0
level of education    0
age                   0
sex                   0
predictor_year        0
population_count      0
dtype: int64


,municipality_code,municipality_name,level of education,age,sex,predictor_year,population_count
0,0114,Upplands Väsby,primary and secondary education less than 9 ye...,25-34 years,men,2012,64
1,0114,Upplands Väsby,primary and secondary education less than 9 ye...,25-34 years,women,2012,76
2,0114,Upplands Väsby,primary and secondary education less than 9 ye...,35-44 years,men,2012,67
3,0114,Upplands Väsby,primary and secondary education less than 9 ye...,35-44 years,women,2012,71
4,0114,Upplands Väsby,primary and secondary education less than 9 ye...,45-54 years,men,2012,59


In [44]:
# Sum population counts across the included age groups and both sexes
# for each municipality, year and education level.

education_grouped = (
    education_long
    .groupby(
        [
            "municipality_code",
            "municipality_name",
            "predictor_year",
            "level of education"
        ],
        as_index=False
    )["population_count"]
    .sum()
)

In [45]:
# Inspect the aggregated education counts.

print("Shape:", education_grouped.shape)

education_grouped.head()

Shape: (9280, 5)


,municipality_code,municipality_name,predictor_year,level of education,population_count
0,0114,Upplands Väsby,2012,no information about level of educational atta...,460
1,0114,Upplands Väsby,2012,post-graduate education (ISCED97 6),178
2,0114,Upplands Väsby,2012,post-secondary education 3 years or more (ISCE...,4139
3,0114,Upplands Väsby,2012,"post-secondary education, less than 3 years (I...",3148
4,0114,Upplands Väsby,2012,primary and secondary education 9-10 years (IS...,2330


In [46]:
# Education categories counted as higher education.

higher_education_levels = [
    "post-secondary education, less than 3 years (ISCED97 4+5B)",
    "post-secondary education 3 years or more (ISCED97 5A)",
    "post-graduate education (ISCED97 6)"
]

In [47]:
# Select higher-education categories.

higher_ed = education_grouped[
    education_grouped["level of education"].isin(
        higher_education_levels
    )
]


# Sum the higher-education population for each municipality and year.

higher_ed_counts = (
    higher_ed
    .groupby(
        [
            "municipality_code",
            "municipality_name",
            "predictor_year"
        ],
        as_index=False
    )["population_count"]
    .sum()
    .rename(
        columns={
            "population_count": "higher_education_count"
        }
    )
)

In [48]:
# Quality-check the higher-education numerator.

print("Shape:", higher_ed_counts.shape)

higher_ed_counts.head()

Shape: (1160, 4)


,municipality_code,municipality_name,predictor_year,higher_education_count
0,0114,Upplands Väsby,2012,7465
1,0114,Upplands Väsby,2016,8720
2,0114,Upplands Väsby,2020,10332
3,0114,Upplands Väsby,2024,12090
4,0115,Vallentuna,2012,6520


In [49]:
# Exclude people whose education level is unknown.

known_education = education_grouped[
    education_grouped["level of education"]
    != "no information about level of educational attainment"
]

In [50]:
# Calculate the number of people with known education information
# for each municipality and predictor year.

total_known_counts = (
    known_education
    .groupby(
        [
            "municipality_code",
            "municipality_name",
            "predictor_year"
        ],
        as_index=False
    )["population_count"]
    .sum()
    .rename(
        columns={
            "population_count": "known_education_count"
        }
    )
)

In [51]:
print("Known-education table shape:", total_known_counts.shape)

Known-education table shape: (1160, 4)


In [52]:
# Combine numerator and denominator.

education_pct = higher_ed_counts.merge(
    total_known_counts[
        [
            "municipality_code",
            "predictor_year",
            "known_education_count"
        ]
    ],
    on=[
        "municipality_code",
        "predictor_year"
    ],
    how="left"
)


# Calculate higher-education percentage.

education_pct["higher_education_pct"] = (
    education_pct["higher_education_count"]
    /
    education_pct["known_education_count"]
    * 100
)

In [53]:
# Quality-check the derived higher-education indicator.

print("Shape:", education_pct.shape)

print(
    "Unique municipalities:",
    education_pct["municipality_code"].nunique()
)

print("\nRows per predictor year:")
print(
    education_pct["predictor_year"]
    .value_counts()
    .sort_index()
)

print(
    "\nMissing higher-education values:",
    education_pct["higher_education_pct"]
    .isna()
    .sum()
)

print(
    "All percentages between 0 and 100:",
    education_pct["higher_education_pct"]
    .between(0, 100)
    .all()
)

print("\nSummary:")
print(
    education_pct["higher_education_pct"]
    .describe()
)

# One manual municipality check.
education_pct[
    education_pct["municipality_name"] == "Malmö"
]

Shape: (1160, 6)
Unique municipalities: 290

Rows per predictor year:
predictor_year
2012    290
2016    290
2020    290
2024    290
Name: count, dtype: int64

Missing higher-education values: 0
All percentages between 0 and 100: True

Summary:
count    1160.000000
mean       34.337601
std        10.372803
min        18.290735
25%        27.039778
50%        31.545866
75%        39.031321
max        79.179321
Name: higher_education_pct, dtype: float64


,municipality_code,municipality_name,predictor_year,higher_education_count,known_education_count,higher_education_pct
464,1280,Malmö,2012,78820,162812,48.411665
465,1280,Malmö,2016,89051,175051,50.871460
466,1280,Malmö,2020,99929,185687,53.815830
467,1280,Malmö,2024,112950,195144,57.880335


In [54]:
# Add higher-education percentage to the historical modelling dataset.
master_df = master_df.merge(
    education_pct[
        [
            "municipality_code",
            "predictor_year",
            "higher_education_pct"
        ]
    ],
    on=["municipality_code", "predictor_year"],
    how="left"
)

In [55]:
# Quality-check the historical dataset after adding education.

print("Shape:", master_df.shape)

print(
    "Missing education values:",
    master_df["higher_education_pct"]
    .isna()
    .sum()
)

print(
    "Duplicate municipality-election rows:",
    master_df.duplicated(
        subset=[
            "municipality_code",
            "election_year"
        ]
    ).sum()
)

master_df[
    master_df["municipality_name"] == "Malmö"
]

Shape: (870, 9)
Missing education values: 0
Duplicate municipality-election rows: 0


,municipality_code,municipality_name,election_year,predictor_year,turnout_pct,previous_turnout_pct,median_income_ksek,registered_unemployed_pct,higher_education_pct
348,1280,Malmö,2014,2012,79.7,79.6,362.3,18.8,48.411665
349,1280,Malmö,2018,2016,82.0,79.7,412.1,19.8,50.871460
350,1280,Malmö,2022,2020,77.2,82.0,427.3,21.2,53.815830


## 6. Prepare Average Age

Average municipal population age is prepared for the predictor years
2012, 2016, 2020 and 2024.

The data are reshaped to long format and merged with the historical modelling
dataset using municipality code and predictor year.

In [56]:
# Load municipal average-age data from the raw-data folder.

age = pd.read_csv(
    "../data/raw/scb_average_age_2012_2024.csv",
    encoding="latin1",
    skiprows=2
)

In [57]:
# Split municipality code and name from the SCB region field.

age[["municipality_code", "municipality_name"]] = (
    age["region"]
    .str.split(" ", n=1, expand=True)
)

In [58]:
# Remove columns not required for this project.
age = age.drop(
    columns=["region", "sex"]
)

In [59]:
# Reshape average-age data from wide to long format.

age_long = age.melt(
    id_vars=["municipality_code", "municipality_name"],
    value_vars=["2012", "2016", "2020", "2024"],
    var_name="predictor_year",
    value_name="average_age"
)

In [60]:
# Convert predictor year and average age to numeric types.

age_long["predictor_year"] = (
    age_long["predictor_year"]
    .astype(int)
)

age_long["average_age"] = pd.to_numeric(
    age_long["average_age"],
    errors="coerce"
)

In [61]:
# Quality-check the long-format average-age dataset.

print("Shape:", age_long.shape)

print("\nRows per predictor year:")
print(
    age_long["predictor_year"]
    .value_counts()
    .sort_index()
)

print(
    "\nMissing average-age values:",
    age_long["average_age"]
    .isna()
    .sum()
)

age_long.head()

Shape: (1160, 4)

Rows per predictor year:
predictor_year
2012    290
2016    290
2020    290
2024    290
Name: count, dtype: int64

Missing average-age values: 0


,municipality_code,municipality_name,predictor_year,average_age
0,0114,Upplands Väsby,2012,39.4
1,0115,Vallentuna,2012,37.8
2,0117,Österåker,2012,39.4
3,0120,Värmdö,2012,38.4
4,0123,Järfälla,2012,39.6


In [62]:
# Add average age to the historical modelling dataset.

master_df = master_df.merge(
    age_long[
        [
            "municipality_code",
            "predictor_year",
            "average_age"
        ]
    ],
    on=["municipality_code", "predictor_year"],
    how="left"
)

In [63]:
# Quality-check the historical dataset after adding average age.

print("Shape:", master_df.shape)

print(
    "Missing average-age values:",
    master_df["average_age"]
    .isna()
    .sum()
)

print(
    "Duplicate municipality-election rows:",
    master_df.duplicated(
        subset=[
            "municipality_code",
            "election_year"
        ]
    ).sum()
)

# One manual municipality check.
master_df[
    master_df["municipality_name"] == "Malmö"
][
    [
        "election_year",
        "predictor_year",
        "average_age"
    ]
]

Shape: (870, 10)
Missing average-age values: 0
Duplicate municipality-election rows: 0


,election_year,predictor_year,average_age
348,2014,2012,38.8
349,2018,2016,38.5
350,2022,2020,38.5


## 7. Prepare Population Density

Population density is prepared for the predictor years 2012, 2016, 2020
and 2024.

The data are reshaped to long format and merged with the historical modelling
dataset using municipality code and predictor year.

Raw population density is retained for interpretation. A logarithmic
transformation is considered later because the distribution is strongly
right-skewed.

In [64]:

# Load municipal population-density data from the raw-data folder.

density = pd.read_csv(
    "../data/raw/scb_population_density_2012_2024.csv",
    encoding="latin1",
    skiprows=2
)

In [65]:
# Inspect the raw population-density dataset.

print("Shape:", density.shape)

print("\nColumns:")
print(density.columns.tolist())

density.head()

Shape: (290, 6)

Columns:
['region', 'sex', '2012', '2016', '2020', '2024']


,region,sex,2012,2016,2020,2024
0,0114 Upplands Väsby,total,542.3,585.1,629.1,667.6
1,0115 Vallentuna,total,87.1,91.6,95.4,97.7
2,0117 Österåker,total,128.9,138.6,149.3,158.6
3,0120 Värmdö,total,87.9,93.7,102.6,104.4
4,0123 Järfälla,total,1267.6,1383.5,1510.9,1645.4


In [66]:
# Split municipality code and municipality name
# from the SCB region field.

density[["municipality_code", "municipality_name"]] = (
    density["region"]
    .str.split(" ", n=1, expand=True)
)

In [67]:
# Remove columns not required for this project.
density = density.drop(
    columns=["region", "sex"]
)

In [68]:
# Reshape population density from wide to long format.

density_long = density.melt(
    id_vars=["municipality_code", "municipality_name"],
    value_vars=["2012", "2016", "2020", "2024"],
    var_name="predictor_year",
    value_name="population_density"
)

In [69]:
# Convert predictor year and density values to numeric types.
density_long["predictor_year"] = (
    density_long["predictor_year"].astype(int)
)

density_long["population_density"] = pd.to_numeric(
    density_long["population_density"],
    errors="coerce"
)

In [70]:
# Quality-check the long-format population-density dataset.

print("Shape:", density_long.shape)

print("\nRows per predictor year:")
print(
    density_long["predictor_year"]
    .value_counts()
    .sort_index()
)

print(
    "\nMissing density values:",
    density_long["population_density"]
    .isna()
    .sum()
)

density_long.head()

Shape: (1160, 4)

Rows per predictor year:
predictor_year
2012    290
2016    290
2020    290
2024    290
Name: count, dtype: int64

Missing density values: 0


,municipality_code,municipality_name,predictor_year,population_density
0,0114,Upplands Väsby,2012,542.3
1,0115,Vallentuna,2012,87.1
2,0117,Österåker,2012,128.9
3,0120,Värmdö,2012,87.9
4,0123,Järfälla,2012,1267.6


In [71]:
# Add population density to the historical modelling dataset.

master_df = master_df.merge(
    density_long[
        [
            "municipality_code",
            "predictor_year",
            "population_density"
        ]
    ],
    on=["municipality_code", "predictor_year"],
    how="left"
)

In [72]:
# ============================================================
# QUALITY CHECK: COMPLETE HISTORICAL DATASET
# ============================================================

print("Shape:", master_df.shape)

print(
    "Unique municipality codes:",
    master_df["municipality_code"].nunique()
)

print(
    "Unique municipality names:",
    master_df["municipality_name"].nunique()
)

print("\nRows per election:")
print(
    master_df["election_year"]
    .value_counts()
    .sort_index()
)

print("\nElection-to-predictor-year mapping:")
print(
    master_df[
        ["election_year", "predictor_year"]
    ]
    .drop_duplicates()
    .sort_values("election_year")
)

print("\nMissing values:")
print(
    master_df.isna().sum()
)

print(
    "\nDuplicate municipality-election rows:",
    master_df.duplicated(
        subset=[
            "municipality_code",
            "election_year"
        ]
    ).sum()
)

Shape: (870, 11)
Unique municipality codes: 290
Unique municipality names: 290

Rows per election:
election_year
2014    290
2018    290
2022    290
Name: count, dtype: int64

Election-to-predictor-year mapping:
   election_year  predictor_year
0           2014            2012
1           2018            2016
2           2022            2020

Missing values:
municipality_code            0
municipality_name            0
election_year                0
predictor_year               0
turnout_pct                  0
previous_turnout_pct         0
median_income_ksek           0
registered_unemployed_pct    0
higher_education_pct         0
average_age                  0
population_density           0
dtype: int64

Duplicate municipality-election rows: 0


## 8. Final Historical Dataset Quality Checks

Before saving the historical modelling dataset, the structure, data types,
ranges, missing values and duplicate observations are checked.

The expected dataset contains:

- 290 municipalities,
- 3 elections per municipality,
- 870 municipality-election observations.

In [73]:
# ============================================================
# FINAL QUALITY CHECK: HISTORICAL MODELLING DATASET
# ============================================================

print("Shape:", master_df.shape)

print(
    "Unique municipalities:",
    master_df["municipality_code"].nunique()
)

print(
    "Duplicate municipality-election rows:",
    master_df.duplicated(
        subset=["municipality_code", "election_year"]
    ).sum()
)

print(
    "Duplicate municipality-predictor-year rows:",
    master_df.duplicated(
        subset=["municipality_code", "predictor_year"]
    ).sum()
)

print("\nMunicipality-code lengths:")
print(
    master_df["municipality_code"]
    .astype(str)
    .str.len()
    .value_counts()
)

print("\nData types:")
print(master_df.dtypes)

print("\nRange checks:")
print(
    "Turnout between 0 and 100:",
    master_df["turnout_pct"].between(0, 100).all()
)

print(
    "Previous turnout between 0 and 100:",
    master_df["previous_turnout_pct"].between(0, 100).all()
)

print(
    "Higher education between 0 and 100:",
    master_df["higher_education_pct"].between(0, 100).all()
)

print(
    "Unemployment between 0 and 100:",
    master_df["registered_unemployed_pct"].between(0, 100).all()
)

print("\nMissing values:")
print(master_df.isna().sum())

Shape: (870, 11)
Unique municipalities: 290
Duplicate municipality-election rows: 0
Duplicate municipality-predictor-year rows: 0

Municipality-code lengths:
municipality_code
4    870
Name: count, dtype: int64

Data types:
municipality_code                str
municipality_name                str
election_year                  int64
predictor_year                 int64
turnout_pct                  float64
previous_turnout_pct         float64
median_income_ksek           float64
registered_unemployed_pct    float64
higher_education_pct         float64
average_age                  float64
population_density           float64
dtype: object

Range checks:
Turnout between 0 and 100: True
Previous turnout between 0 and 100: True
Higher education between 0 and 100: True
Unemployment between 0 and 100: True

Missing values:
municipality_code            0
municipality_name            0
election_year                0
predictor_year               0
turnout_pct                  0
previous_turnout_

In [74]:
# Descriptive statistics for the historical modelling variables.
master_df[
    [
        "turnout_pct",
        "previous_turnout_pct",
        "median_income_ksek",
        "registered_unemployed_pct",
        "higher_education_pct",
        "average_age",
        "population_density"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
turnout_pct,870.0,85.958506,2.893326,69.100000,84.42500,86.100000,87.800000,93.900000
previous_turnout_pct,870.0,85.766897,2.820496,70.400000,84.10000,85.900000,87.700000,93.900000
median_income_ksek,870.0,449.042644,79.149468,325.300000,398.85000,429.350000,469.175000,861.700000
registered_unemployed_pct,870.0,14.015632,3.580893,5.300000,11.40000,14.000000,16.600000,25.100000
higher_education_pct,870.0,32.982987,9.993472,18.290735,25.74878,30.376384,37.658829,76.707637
average_age,870.0,43.353218,2.647266,36.500000,41.50000,43.600000,45.200000,51.100000
population_density,870.0,150.520460,542.689899,0.200000,12.32500,27.350000,77.475000,6076.100000


In [75]:
# Manual sanity check for one municipality.

master_df.loc[
    master_df["municipality_name"] == "Malmö",
    [
        "election_year",
        "turnout_pct",
        "previous_turnout_pct"
    ]
]

,election_year,turnout_pct,previous_turnout_pct
348,2014,79.7,79.6
349,2018,82.0,79.7
350,2022,77.2,82.0


In [76]:
master_df.loc[
    master_df["municipality_name"] == "Malmö",
    [
        "election_year",
        "predictor_year",
        "median_income_ksek",
        "registered_unemployed_pct",
        "higher_education_pct",
        "average_age",
        "population_density"
    ]
]

,election_year,predictor_year,median_income_ksek,registered_unemployed_pct,higher_education_pct,average_age,population_density
348,2014,2012,362.3,18.8,48.411665,38.8,1961.8
349,2018,2016,412.1,19.8,50.871460,38.5,2097.6
350,2022,2020,427.3,21.2,53.815830,38.5,2216.9


In [77]:
master_df[
    [
        "turnout_pct",
        "previous_turnout_pct",
        "median_income_ksek",
        "registered_unemployed_pct",
        "higher_education_pct",
        "average_age",
        "population_density"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
turnout_pct,870.0,85.958506,2.893326,69.100000,84.42500,86.100000,87.800000,93.900000
previous_turnout_pct,870.0,85.766897,2.820496,70.400000,84.10000,85.900000,87.700000,93.900000
median_income_ksek,870.0,449.042644,79.149468,325.300000,398.85000,429.350000,469.175000,861.700000
registered_unemployed_pct,870.0,14.015632,3.580893,5.300000,11.40000,14.000000,16.600000,25.100000
higher_education_pct,870.0,32.982987,9.993472,18.290735,25.74878,30.376384,37.658829,76.707637
average_age,870.0,43.353218,2.647266,36.500000,41.50000,43.600000,45.200000,51.100000
population_density,870.0,150.520460,542.689899,0.200000,12.32500,27.350000,77.475000,6076.100000


## 9. Log Transformation of Population Density

Population density is strongly right-skewed because a small number of urban
municipalities have much higher population density than most Swedish
municipalities.

To reduce this skewness, the natural logarithm of population density is used
as the modelling feature:

\[
\text{log population density}
=
\ln(\text{population density})
\]

The original population-density variable is retained for interpretation,
while `log_population_density` is used in the statistical and predictive
models.

In [78]:
# Measure skewness before transformation.

raw_density_skew = master_df["population_density"].skew()

print(
    "Raw population-density skewness:",
    round(raw_density_skew, 3)
)

Raw population-density skewness: 7.401


In [79]:
# Apply natural-log transformation.

master_df["log_population_density"] = np.log(
    master_df["population_density"]
)

In [80]:
# Check skewness after log transformation.

log_density_skew = (
    master_df["log_population_density"].skew()
)

print(
    "Log population-density skewness:",
    round(log_density_skew, 3)
)

Log population-density skewness: 0.107


## 10. Save the Historical Modelling Dataset

The completed historical dataset contains one row for each
municipality-election observation in 2014, 2018 and 2022.

This file is used by the exploratory, statistical and predictive modelling
notebooks.

In [81]:
# Save the completed historical modelling dataset
# in the project's processed-data folder.

master_df.to_csv(
    "../data/processed/master_model_dataset_2014_2022.csv",
    index=False
)

print(
    "Historical modelling dataset saved:",
    master_df.shape
)

Historical modelling dataset saved: (870, 12)


## Construction of the 2026 Forecast Dataset

The historical modelling dataset contains election observations for
2014, 2018, and 2022.

For the prospective 2026 forecast:

- Previous turnout = 2022 Riksdag turnout
- Socioeconomic predictor year = 2024
- Target turnout = 2026 Riksdag turnout, which is not yet known

The 2026 feature dataset therefore contains one row for each of Sweden's
290 municipalities and only information available before the 2026 election.

In [82]:
# Create one row for each municipality for the 2026 forecast.
#
# We take municipality identifiers and actual 2022 turnout.
# The 2022 turnout becomes "previous_turnout_pct" for the 2026 election.

forecast_2026 = master_df.loc[
    master_df["election_year"] == 2022,
    [
        "municipality_code",
        "municipality_name",
        "turnout_pct"
    ]
].copy()


# Rename the 2022 turnout column because, for the 2026 forecast,
# it represents turnout from the previous election.

forecast_2026 = forecast_2026.rename(
    columns={
        "turnout_pct": "previous_turnout_pct"
    }
)


# Add the election we are forecasting.
forecast_2026["election_year"] = 2026


# Socioeconomic predictors for the 2026 forecast come from 2024.
forecast_2026["predictor_year"] = 2024

In [83]:
print("Forecast backbone shape:", forecast_2026.shape)

forecast_2026.head()

Forecast backbone shape: (290, 5)


,municipality_code,municipality_name,previous_turnout_pct,election_year,predictor_year
2,0114,Upplands Väsby,79.5,2026,2024
5,0115,Vallentuna,87.4,2026,2024
8,0117,Österåker,87.3,2026,2024
11,0120,Värmdö,88.4,2026,2024
14,0123,Järfälla,79.4,2026,2024


In [84]:
# Add 2024 median income to each municipality.

forecast_2026 = forecast_2026.merge(
    income_long[
        [
            "municipality_code",
            "predictor_year",
            "median_income_ksek"
        ]
    ],
    on=["municipality_code", "predictor_year"],
    how="left"
)

In [85]:
# Add 2024 registered unemployment.

forecast_2026 = forecast_2026.merge(
    unemployment_long[
        [
            "municipality_code",
            "predictor_year",
            "registered_unemployed_pct"
        ]
    ],
    on=["municipality_code", "predictor_year"],
    how="left"
)

In [86]:
# Add the percentage of the 25–64 population
# with higher education in 2024.

forecast_2026 = forecast_2026.merge(
    education_pct[
        [
            "municipality_code",
            "predictor_year",
            "higher_education_pct"
        ]
    ],
    on=["municipality_code", "predictor_year"],
    how="left"
)

In [87]:
# Check the current structure of the 2026 forecasting dataset.

print("Shape:", forecast_2026.shape)

print("\nColumns:")
print(forecast_2026.columns.tolist())

forecast_2026.head()

Shape: (290, 8)

Columns:
['municipality_code', 'municipality_name', 'previous_turnout_pct', 'election_year', 'predictor_year', 'median_income_ksek', 'registered_unemployed_pct', 'higher_education_pct']


,municipality_code,municipality_name,previous_turnout_pct,election_year,predictor_year,median_income_ksek,registered_unemployed_pct,higher_education_pct
0,0114,Upplands Väsby,79.5,2026,2024,525.3,13.4,46.650718
1,0115,Vallentuna,87.4,2026,2024,647.3,8.0,52.457162
2,0117,Österåker,87.3,2026,2024,668.2,8.0,50.757916
3,0120,Värmdö,88.4,2026,2024,642.6,7.9,50.606139
4,0123,Järfälla,79.4,2026,2024,523.9,14.0,51.965445


In [88]:
# Add 2024 average age.

forecast_2026 = forecast_2026.merge(
    age_long[
        [
            "municipality_code",
            "predictor_year",
            "average_age"
        ]
    ],
    on=[
        "municipality_code",
        "predictor_year"
    ],
    how="left"
)

In [89]:
# Add 2024 population density.

forecast_2026 = forecast_2026.merge(
    density_long[
        [
            "municipality_code",
            "predictor_year",
            "population_density"
        ]
    ],
    on=[
        "municipality_code",
        "predictor_year"
    ],
    how="left"
)

In [90]:
# Same transformation used in the historical modelling dataset.

forecast_2026["log_population_density"] = np.log(
    forecast_2026["population_density"]
)

In [91]:
# Arrange the 2026 forecast variables in a clear order.

forecast_2026 = forecast_2026[
    [
        "municipality_code",
        "municipality_name",
        "previous_turnout_pct",
        "election_year",
        "predictor_year",
        "median_income_ksek",
        "registered_unemployed_pct",
        "higher_education_pct",
        "average_age",
        "population_density",
        "log_population_density"
    ]
]

In [92]:
# ============================================================
# FINAL QUALITY CHECK: 2026 FORECAST DATASET
# ============================================================

print("Shape:", forecast_2026.shape)

print(
    "Unique municipalities:",
    forecast_2026["municipality_code"].nunique()
)

print(
    "Duplicate municipalities:",
    forecast_2026["municipality_code"]
    .duplicated()
    .sum()
)

print("\nElection year:")
print(
    forecast_2026["election_year"]
    .value_counts()
)

print("\nPredictor year:")
print(
    forecast_2026["predictor_year"]
    .value_counts()
)

print("\nMissing values:")
print(
    forecast_2026.isna().sum()
)

Shape: (290, 11)
Unique municipalities: 290
Duplicate municipalities: 0

Election year:
election_year
2026    290
Name: count, dtype: int64

Predictor year:
predictor_year
2024    290
Name: count, dtype: int64

Missing values:
municipality_code            0
municipality_name            0
previous_turnout_pct         0
election_year                0
predictor_year               0
median_income_ksek           0
registered_unemployed_pct    0
higher_education_pct         0
average_age                  0
population_density           0
log_population_density       0
dtype: int64


In [93]:
# Automated quality checks for the 2026 forecast dataset.

assert forecast_2026.shape == (290, 11)
assert forecast_2026["municipality_code"].nunique() == 290
assert forecast_2026["municipality_code"].duplicated().sum() == 0
assert forecast_2026.isna().sum().sum() == 0
assert set(forecast_2026["election_year"]) == {2026}
assert set(forecast_2026["predictor_year"]) == {2024}

print("All 2026 forecast-data quality checks passed.")

All 2026 forecast-data quality checks passed.


In [94]:
# Save the completed 2026 forecast feature dataset
# in the project's processed-data folder.

forecast_2026.to_csv(
    "../data/processed/forecast_features_2026.csv",
    index=False
)

print(
    "2026 forecast feature dataset saved:",
    forecast_2026.shape
)

2026 forecast feature dataset saved: (290, 11)
